# Demo 5 · Search Aggregation：搜出来就分好面

官方 demo `movie-discovery`：90 部合成电影，用自然语言描述想看的故事，向量检索命中后**服务端直接按分面聚合**（类型/年代/地区），每个桶带 count 和桶内 Top 命中——不用把 90 条全拉回来自己在 Python 里 groupby。
数据（CC0）：`https://raw.githubusercontent.com/zc277584121/milvus3-demos/main/demos/movie-discovery/data/movie-catalog-r1/movies.json`
嵌入：synopsis 用文本嵌入 API（同前两节，环境变量 EMB_KEY）。

In [1]:
%pip install -q pymilvus==3.0.2 requests numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: C:\Users\liuxiao\GenericAgent\temp\milvus3_lab\venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import json, os, re, requests, numpy as np

BASE = "https://raw.githubusercontent.com/zc277584121/milvus3-demos/main/demos/movie-discovery/data/movie-catalog-r1/"
EMB_EP = os.environ.get("EMB_EP", "https://dashscope.aliyuncs.com/compatible-mode/v1")
EMB_KEY = os.environ.get("EMB_KEY") or input("粘贴嵌入 API key: ")
EMB_MODEL = os.environ.get("EMB_MODEL", "text-embedding-v4")

movies = json.loads(requests.get(BASE + "movies.json", timeout=60).text)["movies"]
presets = json.loads(requests.get(BASE + "presets.json", timeout=60).text)["presets"]
print("电影:", len(movies), "| 预设查询:", [p["id"] for p in presets])
print("字段:", [k for k in movies[0] if k != "poster_palette"])

def embed(texts, batch=10):
    vecs = []
    for i in range(0, len(texts), batch):
        r = requests.post(EMB_EP.rstrip("/") + "/embeddings",
                          headers={"Authorization": f"Bearer {EMB_KEY}"},
                          json={"model": EMB_MODEL, "input": texts[i:i + batch], "dimensions": 1024},
                          timeout=60)
        r.raise_for_status()
        vecs += [d["embedding"] for d in r.json()["data"]]
    return np.asarray(vecs, dtype=np.float32)

电影: 90 | 预设查询: ['lighthearted-adventure', 'cozy-animated-story', 'quiet-town-drama', 'overnight-train-mystery', 'optimistic-space-story']
字段: ['movie_id', 'title', 'synopsis', 'primary_genre', 'release_year', 'decade', 'region', 'runtime_minutes', 'poster_path']


## 1. 建集合：分面字段是普通标量列，聚合按它们分组

In [3]:
from pymilvus import MilvusClient, DataType

client = MilvusClient(uri="http://127.0.0.1:19530")
COLL = "demo5_movies"
if client.has_collection(COLL):
    client.drop_collection(COLL)

schema = client.create_schema(auto_id=False, enable_dynamic_field=False)
schema.add_field("movie_id", DataType.VARCHAR, is_primary=True, max_length=16)
schema.add_field("title", DataType.VARCHAR, max_length=256)
schema.add_field("synopsis", DataType.VARCHAR, max_length=2048)
schema.add_field("primary_genre", DataType.VARCHAR, max_length=32)
schema.add_field("decade", DataType.VARCHAR, max_length=16)
schema.add_field("region", DataType.VARCHAR, max_length=64)
schema.add_field("release_year", DataType.INT64)
schema.add_field("runtime_minutes", DataType.INT64)
schema.add_field("embedding", DataType.FLOAT_VECTOR, dim=1024)

index_params = client.prepare_index_params()
index_params.add_index(field_name="embedding", index_type="HNSW", metric_type="COSINE",
                       params={"M": 16, "efConstruction": 128})
client.create_collection(COLL, schema=schema, index_params=index_params)

vecs = embed([m["synopsis"] for m in movies])
rows = []
for m, v in zip(movies, vecs):
    rows.append({"movie_id": m["movie_id"], "title": m["title"], "synopsis": m["synopsis"],
                 "primary_genre": m["primary_genre"], "decade": m["decade"], "region": m["region"],
                 "release_year": m["release_year"], "runtime_minutes": m["runtime_minutes"],
                 "embedding": v.tolist()})
client.insert(COLL, rows)
client.flush(COLL)
client.load_collection(COLL)
print("inserted:", client.get_collection_stats(COLL))

inserted: {'row_count': 90}


## 2. 聚合搜索：一次请求，桶 + 桶内 Top 命中

In [4]:
from pymilvus import SearchAggregation, TopHits

preset = presets[0]
print("查询:", preset["text"])
qvec = embed([preset["text"]])[0]

agg = SearchAggregation(fields=["primary_genre"], size=10,
                        top_hits=TopHits(size=3, sort=[{"_score": "desc"}]))
res = client.search(COLL, data=[qvec.tolist()], anns_field="embedding",
                    search_params={"metric_type": "COSINE", "params": {"ef": 64}},
                    limit=1, output_fields=["movie_id"], search_aggregation=agg)

def bucket_label(b):
    k = b.key
    return k[0]["value"] if isinstance(k, list) else k

for b in res.agg_buckets[0]:
    tops = [(h.pk, round(float(h.score), 4)) for h in b.hits]
    print(f"{bucket_label(b):10} count={b.count:2} top:", tops)

查询: A lighthearted adventure with ordinary people


2026-11-05 16:14:16,782 [WARNING][search]: search_aggregation is set; search limit=1 is ignored and bucket count comes from SearchAggregation.size (grpc_handler.py:1415)


Comedy     count= 3 top: [('m0028', 0.4578), ('m0020', 0.4526), ('m0026', 0.4451)]
Animation  count= 3 top: [('m0032', 0.4504), ('m0040', 0.4204), ('m0031', 0.4138)]
Adventure  count= 3 top: [('m0001', 0.4452), ('m0002', 0.4417), ('m0011', 0.3672)]
Thriller   count= 3 top: [('m0090', 0.3969), ('m0079', 0.3585), ('m0076', 0.3063)]
Sci-Fi     count= 3 top: [('m0072', 0.3813), ('m0063', 0.3288), ('m0073', 0.3149)]
Drama      count= 3 top: [('m0047', 0.3713), ('m0052', 0.3591), ('m0049', 0.3574)]


## 3. 分面 + filter 组合：先按标量条件圈定，再在圈定范围内分面

In [5]:
agg2 = SearchAggregation(fields=["decade"], size=10,
                         top_hits=TopHits(size=2, sort=[{"_score": "desc"}]))
res2 = client.search(COLL, data=[qvec.tolist()], anns_field="embedding",
                     search_params={"metric_type": "COSINE", "params": {"ef": 64}},
                     limit=1, output_fields=["movie_id"], search_aggregation=agg2,
                     filter='primary_genre == "Comedy"')
for b in res2.agg_buckets[0]:
    print(f"{bucket_label(b):8} count={b.count:2}", [(h.pk, round(float(h.score), 3)) for h in b.hits])

2026-11-05 16:14:16,800 [WARNING][search]: search_aggregation is set; search limit=1 is ignored and bucket count comes from SearchAggregation.size (grpc_handler.py:1415)


2000s    count= 2 [('m0028', 0.458), ('m0024', 0.425)]
2010s    count= 2 [('m0020', 0.453), ('m0026', 0.445)]
2020s    count= 2 [('m0016', 0.444), ('m0017', 0.412)]
1980s    count= 1 [('m0025', 0.431)]
1990s    count= 1 [('m0030', 0.384)]


## 4. 四条实现约束

1. **桶数由 `SearchAggregation(size=...)` 决定**：设置 `search_aggregation` 后，`limit` 不再影响桶数（pymilvus 会打印 warning 提醒）；
2. **`bucket.key` 是属性且为列表**：多字段聚合时每个元素对应一个字段，取值 `b.key[0]["value"]`；
3. **`count` 与 `hits` 含义不同**：`count` 是桶内命中总数，`hits` 只有 `TopHits(size=...)` 条；
4. **聚合发生在过滤后的召回集上**：`filter` 先圈定范围，分面统计的是圈定范围内的分布，不是全库分布。


In [6]:
client.drop_collection(COLL)
print("done")

done


这条链验证了三件事：

- 同一查询按 genre 分面：6 个桶，每个桶带 count 和桶内 top 命中，服务端一次算完；
- 加 `filter` 后分面随圈定范围收缩（Comedy 只剩 5 个 decade 桶）；
- 官方 demo 前端把这些桶画成可点击的分面卡片，JSON 就是它背后的数据。
